In [4]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental
%pip install -U ddgs


   ---------------------- ----------------- 4/7 [fake-useragent]
   ---------------------------------------- 7/7 [ddgs]

Note: you may need to restart the kernel to use updated packages.


### Built-in Tool - DuckDuckGo Search

In [5]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('top news in india today')

print(results)

The Hindu Newspaper: Get latest News on Politics, Sports, Business, Arts, Entertainment and trending news Videos from The Hindu 12 hours ago - MEA spokesperson Randhir Jaiswal said the state of border affairs will have a bearing on broader India-China bilateral ties. ... Bharti was repatriated to his parent cadre “on the request of Ministry of Home Affairs with imposition of extended ‘cooling off’”, the ACC order said. ... Rahul Gandhi demands Home Minister explain who ordered action; Nadda criticises LoP, says he only wants anarchy, not dialogue. ... Punjabi-Spanish fusion of Talibelico in collaboration with Punjabi singer Harveer Singh is garnering millions of views online. 18 hours ago - AajTak: Hindi news (हिंदी समाचार) website, watch live tv coverages, Latest Khabar, Breaking news in Hindi of India, World, Sports, business, film and Entertainment. आज तक पर पढ़ें ताजा समाचार देश और दुनिया ... 1 day ago - National news and headlines from within or around India with live updates on m

In [6]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


### Built-in Tool - Shell Tool

In [7]:
from langchain_community.tools import ShellTool
shell_tool = ShellTool()
results = shell_tool.invoke('whoami')
print(results)

Executing command:
 whoami
anoop\anoop



d:\Computer Courses\Agenetic-Ai\langchain_venv\Lib\site-packages\langchain_community\tools\shell\tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


### Custom Tools

In [12]:
from langchain_core.tools import tool

In [8]:
# Step 1 - Create a function
def multiply(a,b):
    '''Multiply two numbers'''
    return a*b

In [9]:
# Step 2 - Add type hints
def multiply(a:int, b: int) -> int:
    '''Multiply two numbers'''
    return a*b

In [13]:
# Step 3 - add tool decorator

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [14]:
result = multiply.invoke({'a':3, 'b':5})

In [15]:
print(result)

15


In [16]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [17]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


### Method 2 - Using StructuredTOol

In [19]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [20]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description='The first number to add')
    b: int = Field(required=True, description='The second number to add')

C:\Users\anoop\AppData\Local\Temp\ipykernel_28308\3455387981.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description='The first number to add')
C:\Users\anoop\AppData\Local\Temp\ipykernel_28308\3455387981.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description='The second number to add')


In [21]:
def multiply_func(a: int, b: int) -> int:
    return a*b

In [22]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name='multiply',
    description='Multiply two numbers',
    args_schema = MultiplyInput
)

In [23]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


### Method 3 - Using BaseTool Class

In [26]:
from langchain_core.tools import BaseTool
from typing import Type

In [25]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

C:\Users\anoop\AppData\Local\Temp\ipykernel_28308\908171234.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\anoop\AppData\Local\Temp\ipykernel_28308\908171234.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


In [28]:
class MultiplyTool(BaseTool):
    name: str = 'multiply'
    description: str = 'Multiply two numbers'

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a*b

In [29]:
multiply_tool = MultiplyTool()

In [30]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


### Toolkit

In [31]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a+b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a*b

In [32]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [34]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, '=>', tool.description)


add => Add two numbers
multiply => Multiply two numbers
